# Lab 04.2 — Logitech C270 USB Webcam

## Objectives

- capture your own image with the already tested Logitech C270;
- allow automatic exposure to settle;
- display frames in Jupyter and release the camera reliably.

Connect the C270 to the Zybo USB host port. Run the cells in order. The verified device is `/dev/video0`; there is no need to repeat camera installation.

## 1. Optional device check

Run this only if the capture below fails. Multiple `/dev/video*` nodes do not necessarily represent usable image streams. Keep `/dev/video0` unless device inspection shows that the camera has moved.

In [ ]:
!lsusb
!ls -l /dev/video* 2>/dev/null || echo "No video device found"
!fuser -v /dev/video0


## Camera setup — provided code

Run this helper cell once in this notebook. It uses the previously verified **V4L2 / `/dev/video0` → 20 warm-up frames → capture → inline JPEG display → release** sequence. The requested resolution is 640×480; use `frame.shape` to check the actual resolution.

The `with camera_session()` block releases the camera through `finally`, including on Python errors or a normal Jupyter interrupt. Only one notebook/kernel should use the camera at a time. If Linux assigns a different capture device, change **CAMERA_DEVICE in each notebook you use**; notebook variables are not shared.

The helper is supplied; you do not need to implement it. Time limits are checked between reads: a USB driver call that blocks can delay an interrupt or timeout.

In [ ]:
import cv2
import numpy as np
import time
from contextlib import contextmanager
from IPython.display import display, Image, clear_output

CAMERA_DEVICE = "/dev/video0"
WARMUP_FRAMES = 20

def read_frame(cap):
    ok, frame = cap.read()
    if not ok or frame is None or frame.size == 0:
        raise RuntimeError(
            "No camera frame. Check the USB connection and close other camera notebooks."
        )
    return frame

@contextmanager
def camera_session():
    """Provided helper: open, warm up, and always release the camera."""
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError(
                "Cannot open " + CAMERA_DEVICE +
                ". Check the device and shut down other kernels using the camera."
            )
        # Request a modest resolution; the driver may negotiate another one.
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        warmup_start = time.monotonic()
        for _ in range(WARMUP_FRAMES):
            read_frame(cap)  # Discard frames while exposure settles.
            if time.monotonic() - warmup_start > 10:
                raise RuntimeError("Camera warm-up is too slow. Check the USB connection.")
        yield cap
    finally:
        cap.release()
        print("Camera released.")

def show_bgr(frame, replace=False):
    ok, jpg = cv2.imencode(".jpg", frame)
    if not ok:
        raise RuntimeError("JPEG encoding failed.")
    if replace:
        clear_output(wait=True)
    display(Image(data=jpg.tobytes()))


## 2. Capture one frame

Hold a colored object in front of the camera before running this cell. The first 20 frames are discarded. The camera is closed before the image is displayed; rerunning this cell starts a fresh capture.

In [ ]:
with camera_session() as cap:
    frame = read_frame(cap)

print("Actual frame shape (height, width, channels):", frame.shape)
print("Data type:", frame.dtype)
show_bgr(frame)


## 3. Short live preview

Move the object while this runs. The preview stops after 60 frames or approximately 10 seconds after warm-up, whichever limit is reached first. You can also use **Kernel → Interrupt**. Rerun the cell for another preview.

In [ ]:
with camera_session() as cap:
    start = time.monotonic()
    for _ in range(60):
        if time.monotonic() - start >= 10:
            break
        frame = read_frame(cap)
        show_bgr(frame, replace=True)
        time.sleep(0.03)


## Short exercise

1. Capture one frame with a colored object clearly visible.
2. Record the actual image resolution.
3. Run the capture again and confirm that the camera reopens.

## If the camera is unavailable

- Shut down any earlier notebook kernel using the camera; closing a browser tab alone may leave its kernel running.
- In an older notebook that opened `cap` without cleanup, execute `cap.release()` in that same kernel.
- Use the optional `fuser` check to identify a process holding `/dev/video0`; empty output normally means no holder was found (subject to permissions).
- If a kernel does not respond to Interrupt because a driver read is blocked, shut down/restart that kernel. If necessary reconnect the camera, check its device path, then rerun the helper cell.
- Do not start a second capture while the first is running. No `cv2.imshow()` or desktop window is needed.